# Q13: K-Means Clustering
**Topic:** Coding-question | **Difficulty:** Medium | **Time:** 20 min
**Sheet:** Grind75ML

---

## Question
Implement K-Means Clustering from scratch.

## Theory

**K-Means** partitions $n$ data points into $k$ clusters by minimizing within-cluster variance:

$$\text{argmin}_{C} \sum_{i=1}^{k}\sum_{x \in C_i} \|x - \mu_i\|^2$$

**Algorithm (Lloyd's):**
1. **Initialize** $k$ centroids (random or K-Means++)
2. **Assign** each point to the nearest centroid
3. **Update** each centroid = mean of its assigned points
4. **Repeat** until convergence (assignments don't change)

**K-Means++ Initialization:** Pick first centroid randomly, then pick each subsequent centroid with probability proportional to $d^2$ (distance to nearest existing centroid). This avoids poor initializations.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


class KMeans:
    """K-Means clustering with K-Means++ initialization."""
    
    def __init__(self, k: int, max_iters: int = 100, tol: float = 1e-6):
        self.k = k
        self.max_iters = max_iters
        self.tol = tol
        self.centroids: np.ndarray = None
        self.labels: np.ndarray = None
        self.inertia: float = 0.0
    
    def _init_centroids_plus_plus(self, X: np.ndarray) -> np.ndarray:
        """K-Means++ initialization for better starting centroids."""
        n = X.shape[0]
        centroids = [X[np.random.randint(n)]]
        
        for _ in range(1, self.k):
            # Distance to nearest centroid
            dists = np.min([np.sum((X - c)**2, axis=1) for c in centroids], axis=0)
            # Probability proportional to d^2
            probs = dists / dists.sum()
            centroids.append(X[np.random.choice(n, p=probs)])
        
        return np.array(centroids)
    
    def fit(self, X: np.ndarray) -> 'KMeans':
        """Fit K-Means to data."""
        self.centroids = self._init_centroids_plus_plus(X)
        
        for iteration in range(self.max_iters):
            # Assign: each point to nearest centroid
            distances = np.array([np.sum((X - c)**2, axis=1) for c in self.centroids]).T
            self.labels = np.argmin(distances, axis=1)
            
            # Update: centroids = mean of assigned points
            new_centroids = np.array([
                X[self.labels == i].mean(axis=0) if np.sum(self.labels == i) > 0
                else self.centroids[i]
                for i in range(self.k)
            ])
            
            # Check convergence
            shift = np.sum((new_centroids - self.centroids)**2)
            self.centroids = new_centroids
            if shift < self.tol:
                print(f"Converged at iteration {iteration + 1}")
                break
        
        self.inertia = sum(
            np.sum((X[self.labels == i] - self.centroids[i])**2)
            for i in range(self.k)
        )
        return self
    
    def predict(self, X: np.ndarray) -> np.ndarray:
        """Assign new points to nearest centroid."""
        distances = np.array([np.sum((X - c)**2, axis=1) for c in self.centroids]).T
        return np.argmin(distances, axis=1)

In [ ]:
# --- Generate sample data ---
np.random.seed(42)
from sklearn.datasets import make_blobs

X, y_true = make_blobs(n_samples=300, centers=4, cluster_std=0.8, random_state=42)

kmeans = KMeans(k=4)
kmeans.fit(X)

print(f"Inertia: {kmeans.inertia:.2f}")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(X[:, 0], X[:, 1], c=y_true, cmap='viridis', s=20, alpha=0.7)
axes[0].set_title('Ground Truth')

axes[1].scatter(X[:, 0], X[:, 1], c=kmeans.labels, cmap='viridis', s=20, alpha=0.7)
axes[1].scatter(kmeans.centroids[:, 0], kmeans.centroids[:, 1], c='red', marker='X', s=200, edgecolors='black')
axes[1].set_title('K-Means Clustering')

plt.tight_layout()
plt.show()

In [ ]:
# --- Elbow Method: finding optimal K ---
inertias = []
K_range = range(1, 10)

for k in K_range:
    km = KMeans(k=k)
    km.fit(X)
    inertias.append(km.inertia)

plt.figure(figsize=(8, 4))
plt.plot(K_range, inertias, 'bo-')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia')
plt.title('Elbow Method')
plt.xticks(list(K_range))
plt.grid(True)
plt.show()

## Key Takeaways
- **K-Means++** initialization nearly always outperforms random — use it by default
- **Elbow Method** helps choose K — look for the point where inertia decreases sharply then levels off
- K-Means assumes **spherical, equal-size clusters** — fails on non-convex shapes
- Always run multiple times (different random seeds) and pick lowest inertia